# Error Handling & Recovery

*Notebook 08*

Tools fail. APIs time out. Data is missing. An agent without error handling crashes on the first unexpected input — an agent with error handling keeps going.

<br>
<br>

**Topics:**
- What happens when a tool raises an exception
- `try/except` patterns inside `@function_tool`
- Returning useful error messages instead of crashing
- Retry patterns with limits
- Fallback strategies — alternative tools and graceful degradation
- Wrapping `Runner.run()` for API-level failures

---

## 🔧 Setup

In [ ]:
import time
import random
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner, function_tool

MODEL = "gpt-5-mini"

print("✅ Ready!")

---

## 🎯 The Problem

Failures happen at multiple layers — tools and the API call itself.

Error handling is the difference between a demo and a production tool.

---

## 💥 Part 1: What Happens Without Error Handling

By default, the SDK catches tool exceptions and returns a generic error message to the model. We disable that default here with `failure_error_function=None` so the raw exception crashes the run — making the failure mode visible. Part 2 will write proper handling that gives the agent something more useful than the generic default.

In [ ]:
# Simulated database — only some records exist
PRODUCT_DB = {
    "P001": {"name": "Wireless Headphones", "price": 149.99, "stock": 23},
    "P002": {"name": "Phone Case", "price": 19.99, "stock": 0},
    "P003": {"name": "Laptop Stand", "price": 49.99, "stock": 7}
}


@function_tool(failure_error_function=None)  # Demo-only: surfaces the raw exception. Don't set this on real tools.
def lookup_product_fragile(product_id: str) -> str:
    """Look up product details by ID."""
    product = PRODUCT_DB[product_id]  # KeyError if product not found
    return f"{product['name']}: ${product['price']:.2f} ({product['stock']} in stock)"


fragile_agent = Agent(
    name="FragileAgent",
    instructions="Look up product information using the available tool.",
    model=MODEL,
    tools=[lookup_product_fragile]
)

# --------------------------------------------------------------
print("✅ Fragile agent ready — intentionally broken")

#### What Happens on a Bad Input

In [ ]:
print("💥 UNHANDLED EXCEPTION DEMO")
print("=" * 60)

try:

    result = await Runner.run(fragile_agent, input="What's the price of product P999?")

    print(result.final_output)

except Exception as e:
    print(f"❌ Run failed: {type(e).__name__}: {e}")
    print("   The entire agent run crashed — user gets nothing")

print("=" * 60)

---

## 🛡️ Part 2: Handling Errors Inside the Tool

By default the SDK already converts tool exceptions into a generic error message for the model. Adding try/except inside the tool gives you control over that message — so the agent can recover with useful, actionable information instead of a vague 'tool errored' string.

In [ ]:
@function_tool
def lookup_product(product_id: str) -> str:
    """Look up product details by ID."""
    try:
        product = PRODUCT_DB[product_id]
        stock_status = "in stock" if product["stock"] > 0 else "out of stock"
        return f"{product['name']}: ${product['price']:.2f} ({product['stock']} units {stock_status})"
    except KeyError:
        return f"Product '{product_id}' not found. Available products: {', '.join(PRODUCT_DB.keys())}"
    except Exception as e:
        return f"Error looking up product: {e}"


robust_instructions = (
    "Look up product information using the available tool.\n"
    "If a product is not found, tell the user and suggest alternatives."
)

robust_agent = Agent(
    name="RobustAgent",
    instructions=robust_instructions,
    model=MODEL,
    tools=[lookup_product]
)

# --------------------------------------------------------------
print("✅ Robust agent ready")

#### Valid Product vs Missing Product

In [ ]:
print("🛡️ ROBUST TOOL DEMO")
print("=" * 60)

for query in [
    "What's the price of product P001?",
    "What's the price of product P999?"
]:
    print(f"\n🙋 {query}")

    result = await Runner.run(robust_agent, input=query)

    print(f"🤖 {result.final_output}")

print("\n" + "=" * 60)

---

## 🔁 Part 3: Retry Pattern

Some failures are transient — a flaky API, a rate limit, a momentary network issue.

For these, retry with a limit rather than failing immediately.

In [ ]:
# Simulates an unreliable external API — fails 60% of the time
def unreliable_api_call(query: str) -> str:
    """Simulates a flaky external service."""
    if random.random() < 0.6:
        raise ConnectionError("Service temporarily unavailable")
    return f"API result for: {query}"


@function_tool
def fetch_with_retry(query: str) -> str:
    """Fetch data from an external service with automatic retry."""
    max_attempts = 3
    wait_seconds = 1

    for attempt in range(1, max_attempts + 1):
        try:
            result = unreliable_api_call(query)
            print(f"    ✅ Succeeded on attempt {attempt}")
            return result
        except ConnectionError as e:
            print(f"    ⚠️  Attempt {attempt} failed: {e}")
            if attempt < max_attempts:
                time.sleep(wait_seconds)

    return f"Service unavailable after {max_attempts} attempts. Please try again later."


retry_agent = Agent(
    name="RetryAgent",
    instructions="Fetch information using the available tool.",
    model=MODEL,
    tools=[fetch_with_retry]
)

# --------------------------------------------------------------
print("✅ Retry agent ready")

#### Run with Retry

In [ ]:
random.seed(42)  # reproducible demo

print("🔁 RETRY PATTERN DEMO")
print("=" * 60)
print("Fetching from unreliable service (60% failure rate)...\n")

result = await Runner.run(retry_agent, input="Fetch the latest sales report")

print(f"\n🤖 {result.final_output}")
print("=" * 60)

### 💡 Why This Works

The retry loop is entirely inside the tool — the agent never knows a retry happened. In production, use exponential backoff (longer wait after each failure) with jitter (a small random offset, so many clients don't all retry at once) instead of fixed waits. In a real application, also remove the print statements — the pattern inside the loop is what matters.

⚠️ **Security note:** Only retry read-only or idempotent actions (operations that produce the same result whether run once or multiple times). Tools that create orders, send messages, or write records may have partially succeeded — retrying risks duplication.

---

## 🔀 Part 4: Fallback Strategy

When a primary tool fails, a fallback tool provides a degraded but still useful response.

The agent chooses the fallback through its instructions, but the tools still need to return errors in a form the agent can recognize.

In [ ]:
# Primary tool — intentionally always fails to demonstrate fallback behavior
@function_tool
def get_live_weather(location: str) -> str:
    """Get live weather data for a location."""
    # Simulates an API outage — returns an error string so the agent can fall back
    return "Error: Weather API is currently unavailable"


# Fallback tool — cached data, not live — always works
@function_tool
def get_weather_forecast(location: str) -> str:
    """Get a general weather forecast for a location (cached data)."""
    forecasts = {
        "new york": "Partly cloudy, highs in the mid-60s this week",
        "london": "Overcast with occasional rain, highs around 55°F",
        "tokyo": "Clear and sunny, highs in the low 70s"
    }
    location_key = location.lower()
    forecast = forecasts.get(location_key, f"General forecast for {location}: mild conditions expected")
    return f"[Cached forecast] {forecast}"


instructions = (
    "Get weather information using get_live_weather.\n"
    "If get_live_weather fails or returns an error, use get_weather_forecast as a fallback.\n"
    "If both tools fail or return errors, ask the user to clarify their request or try a different location.\n"
    "Always tell the user which source you used and whether the data is live or cached."
)

fallback_agent = Agent(
    name="WeatherAgent",
    instructions=instructions,
    model=MODEL,
    tools=[get_live_weather, get_weather_forecast]
)

# --------------------------------------------------------------
print("✅ Fallback agent ready")

#### Primary Fails, Fallback Succeeds

In [ ]:
print("🔀 FALLBACK STRATEGY DEMO")
print("=" * 60)
print("Primary weather API is down — agent should fall back to cached forecast\n")

result = await Runner.run(fallback_agent, input="What's the weather like in London?")

print(f"🤖 {result.final_output}")
print("=" * 60)

### 💡 Why This Works

The convention from Part 2 holds here: a failing tool returns a clear error string instead of raising. The agent reads that string and follows the instructions to call the fallback — there is no automatic SDK mechanism. Putting recovery logic in the instructions (rather than a Python try/except around the tool calls) means you can change the strategy by editing the prompt as you add or swap tools.

---

## 🔌 Part 5: Runner-Level Recovery

Not all failures happen inside a tool — the run itself can fail.

Wrap `Runner.run()` in a try/except as a safety net for API errors, rate limits, auth failures, and network issues.

In [ ]:
safe_agent = Agent(
    name="SafeAgent",
    instructions="You are a helpful assistant.",
    model=MODEL
)


async def run_safely(agent, user_input: str) -> str:
    """Run an agent with graceful fallback on failure."""
    try:

        result = await Runner.run(agent, input=user_input)

        return result.final_output
    except Exception as e:
        # Catches API errors, auth failures, rate limits, network issues
        error_type = type(e).__name__
        print(f"    ⚠️  Run failed: {error_type}: {e}")
        return "I'm having trouble right now. Please try again in a moment."

# --------------------------------------------------------------
print("✅ SafeAgent and run_safely() ready")

In [ ]:
print("🔌 RUNNER-LEVEL RECOVERY DEMO")
print("=" * 60)

response = await run_safely(safe_agent, "What is the capital of France?")

print(f"\n🤖 {response}")
print("=" * 60)
print("\n💡 In production, log the full error for debugging")
print("     while returning a safe user-facing message.")

---

## 💪 Practice Exercises

### Exercise 1: Robust Calculator

*Covers: structured tool arguments and crash-safe error handling*

Build a calculator tool that handles division by zero, invalid inputs, and overflow gracefully.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Robust Calculator
# --------------------------------------------------------------
# Objective: A calculator tool that never crashes — always returns a useful result or a clear error message.

# TODO 1: Create a @function_tool called safe_calculate
#            Parameters: expression (str) — e.g. "10 / 0" or "2 ** 100"
#            Use eval() to evaluate the expression
#            Catch: ZeroDivisionError, ValueError, SyntaxError, OverflowError
#            Return a descriptive error message for each failure case
#            ⚠️  eval() is unsafe in production — fine for this exercise only

# TODO 2: Create an agent that uses safe_calculate

# TODO 3: Test with these inputs:
#            - "What is 144 / 12?"
#            - "What is 10 / 0?"
#            - "What is this: @#$%?"
#            Print each result — confirm no run crashes

# --- Write your code below this line ---

### Exercise 2: File Reader with Fallback

*Covers: fallback tools — graceful recovery from tool failure*

Build a tool that reads a file, and falls back to a default message if the file doesn't exist.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: File Reader with Fallback
# --------------------------------------------------------------
# Objective: Read a file if it exists, fall back gracefully if it doesn't.

# TODO 1: Create a @function_tool called read_notes
#            Parameter: filename (str)
#            Try to read the file from the current directory
#            If FileNotFoundError: return "No notes file found for: {filename}"
#            If PermissionError: return "Cannot access {filename} — permission denied"

# TODO 2: Create a @function_tool called get_default_notes
#            No parameters — returns a hardcoded default message
#            e.g. "No notes available. Starting with a fresh session."

# TODO 3: Create an agent with both tools
#            Instructions: try read_notes first; if file not found, use get_default_notes

# TODO 4: Test with a filename that doesn't exist
#            Confirm the agent falls back gracefully and tells the user

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

<br>
<br>
**Handle errors inside the tool:**
- Use `try/except` to catch known failure modes

- Return a descriptive string — never let an exception propagate to the agent run

- Catch specific exceptions (`KeyError`, `ConnectionError`) before broad `Exception`

<br>
<br>
**Retry transient failures:**
- Set a `max_attempts` limit — never retry without a ceiling

- Add a short `time.sleep()` between attempts to avoid hammering a struggling service

- Return a clear failure message if all attempts are exhausted

<br>
<br>
**Use fallback strategies:**
- Give the agent a secondary tool and tell it when to use it

- Be transparent with the user about degraded data sources

- Partial information delivered gracefully beats a crash

<br>
<br>
**Wrap Runner.run() for API-level failures:**
- Catch exceptions around `Runner.run()` to handle API errors, rate limits, and network issues

- Return a safe user-facing message — log the full error separately

<br>
<br>
**When something fails, tracing shows you where:**
- Tracing records every tool call, input, and output — covered in Lesson 25

---

## 📍 Next Step

**Notebook 09: Testing & Evaluating Agents**  

Build a golden test set, measure what's actually working, and catch regressions early.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-08-error-handling-and-recovery)

---